<a href="https://colab.research.google.com/github/dA-Wn-7/MindCare/blob/main/FineTuning_version_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup and Configuration

In [ ]:
# @title Install dependencies (Colab / T4 friendly)
!pip -q install --upgrade transformers accelerate datasets peft bitsandbytes huggingface_hub

# Optional: print package versions
import transformers, datasets, peft, torch, accelerate
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)

# T4: prefer fp16; avoid bf16/tf32 issues
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA available?", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 50.0 MB/s eta 0:00:00
transformers: 5.5.0
datasets: 4.8.4
peft: 0.18.1
torch: 2.10.0+cu128
CUDA available? True


In [ ]:
# @title Configuration (edit as needed)
from datetime import datetime

# Pick an instruct-tuned base (Mistral / Llama / Qwen, etc.); demo uses Mistral
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"  # Or any instruct model with a compatible chat template

# Output dir and Hub repo id
HF_OUTPUT_DIR = "mistral7b-qlora-sft-small-v2.1"
HF_REPO_ID = "imnotdawn/mistral7b-qlora-sft-small-v2.1"
# Random seed and subsample sizes (tune for VRAM)
SEED = 42
EMP_SAMP = 3000   # Empathy dataset sample count
THER_SAMP = 3000  # Therapy dataset sample count


# Training hyperparameters (T4-friendly)
EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 2024  # Lower to 512 if VRAM is tight
LOG_STEPS = 50
SAVE_STEPS = 200

In [ ]:
# @title Log in to Hugging Face Hub (interactive)
from huggingface_hub import notebook_login
notebook_login()  # Paste HF token (write access)

# 2. Model and Tokenizer Initialization

In [ ]:
# @title Load tokenizer (chat template & max length)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
# Training max length; Trainer falls back to tokenizer.model_max_length
tokenizer.model_max_length = MAX_SEQ_LEN

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
# @title QLoRA 4-bit quantization (compute dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # nf4 is typical
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16  # T4: use float16 for compute if bfloat16 misbehaves
)

In [ ]:
# @title Load base model (LoRA added later via PEFT)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.bfloat16          # Main dtype (see note above)
)
model.config.use_cache = False  # Disable KV cache during training

# Right after tokenizer load
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS as pad
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Sync pad id into model config
model.config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
# === 3.x: k-bit prep before QLoRA (gradients + checkpoint warnings) ===
from peft import prepare_model_for_kbit_training

# 1) Gradient checkpointing
model.gradient_checkpointing_enable()

# 2) Inputs need grads with checkpointing
model.enable_input_require_grads()

# 3) prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
# Note: if dtype drifts to fp32 vs bnb compute dtype,
# you can sometimes keep only (1)(2) without prepare_model_for_kbit_training.

# 3. Data Loading and Preprocessing

In [ ]:
# @title Load data
from datasets import load_dataset, concatenate_datasets

torch.manual_seed(SEED)

# === A) Empathy: Synthetic_Therapy_Conversations
emp_raw = load_dataset("Mr-Bhaskar/Synthetic_Therapy_Conversations", split="train")
emp_raw = emp_raw.shuffle(seed=SEED).select(range(EMP_SAMP))

def map_emp_to_text(ex):
    user = ex.get("human") or ex.get("Human") or ex.get("human_text")
    assistant = ex.get("ai") or ex.get("AI") or ex.get("assistant_text")

    if not user or not assistant:
        return {"text": None}

    msgs = [
        {"role": "user", "content": str(user)},
        {"role": "assistant", "content": str(assistant)}
    ]

    try:
        # apply_chat_template will automatically add BOS (<s>) by default for Mistral
        # We explicitly set add_eos_token=False as per user's advice to control EOS manually
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
            add_bos_token=True,
            add_eos_token=False # Manually control EOS token for consistency
        )
        if not text.endswith(tokenizer.eos_token):
            text += tokenizer.eos_token

        return {"text": text}
    except Exception as e:
        print(f"Error processing sample: {e}")
        return {"text": None}

emp = emp_raw.map(map_emp_to_text, remove_columns=emp_raw.column_names)
emp = emp.filter(lambda x: x["text"] is not None)
print("Empathy samples:", len(emp))
print(emp[0]["text"][:200])

# === B) Therapy: phr_mental_therapy_dataset (pre-templated text)
ther_raw = load_dataset("vibhorag101/phr_mental_therapy_dataset", split="train")
ther_raw = ther_raw.shuffle(seed=SEED).select(range(THER_SAMP))

# Card shows [INST] style; use as plain text
def pick_text(ex):
    for k in ["text", "content", "prompt"]:
        if isinstance(ex.get(k), str) and len(ex[k].strip()) > 0:
            return {"text": ex[k]}
    txts = [str(v) for v in ex.values() if isinstance(v, str)]
    if txts:
        return {"text": "\n".join(txts)}
    return {"text": None}

ther = ther_raw.map(pick_text, remove_columns=ther_raw.column_names)
ther = ther.filter(lambda x: x["text"] is not None)
print("Therapy samples:", len(ther))
print(ther[0]["text"][:200])

# === Merge (both have text column)
train_dataset = concatenate_datasets([emp, ther]).shuffle(seed=SEED)
print("Train dataset size:", len(train_dataset))

train.csv:   0%|          | 0.00/3.30M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/18.3k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/38 [00:00<?, ? examples/s]

...

data/train-00000-of-00001-50d706348b355a(…):   0%|          | 0.00/211M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/99086 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3000 [00:00<?, ? examples/s]

Therapy samples: 3000
<s>[INST] <<SYS>>
You are a helpful and joyous mental therapy assistant. Always answer as helpfully and cheerfully as possible, while being safe.  Your answers should not include any harmful, unethica
Train dataset size: 5998


In [ ]:
from transformers import DataCollatorForSeq2Seq

def tokenize_and_mask_function(examples):
    """
    针对 Mistral-7B-v0.2 的严格 Loss Masking。
    前提：examples['text'] 已经由 apply_chat_template 处理好，格式为：
    <s>[INST] User Content [/INST] Assistant Content </s>
    """
    text = examples["text"]

    # 1. Tokenize
    # add_special_tokens=False 是因为 text 中已经包含了 <s> 和 </s>
    model_inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        add_special_tokens=False, # 关键：避免重复添加 BOS/EOS
        return_tensors=None
    )

    input_ids = model_inputs["input_ids"]
    labels = [-100] * len(input_ids)

    # 2. 找到 [/INST] 的位置
    # Mistral tokenizer 将 [/INST] 编码为一个或多个 token。
    # 我们查找这个序列在 input_ids 中的最后一次出现。
    inst_end_str = "[/INST]"
    inst_end_tokens = tokenizer.encode(inst_end_str, add_special_tokens=False)

    found_index = -1
    seq_len = len(inst_end_tokens)

    # 逆向查找最后一个 [/INST]，确保我们 mask 的是当前轮的 User，而不是历史中的
    if seq_len > 0 and len(input_ids) >= seq_len:
        for i in range(len(input_ids) - seq_len, -1, -1):
            if input_ids[i:i+seq_len] == inst_end_tokens:
                found_index = i + seq_len # Assistant 回复从 [/INST] 之后开始
                break

    if found_index != -1:
        # 3. 应用 Masking
        # labels[0 : found_index] 保持为 -100 (User part + [INST] tags)
        # labels[found_index : end] 设置为真实 ID (Assistant part + EOS)
        for i in range(found_index, len(input_ids)):
            labels[i] = input_ids[i]
    else:
        # 如果没找到 [/INST]，说明数据格式可能有误。
        # 为了安全，整条数据都 mask 掉（不产生梯度），或者打印警告
        print(f"Warning: Could not find '[/INST]' in sample. Skipping loss for this sample.")
        # labels 保持全 -100

    model_inputs["labels"] = labels
    return model_inputs

# 在 dataset.map 中使用新函数
tokenized = train_dataset.map(
    tokenize_and_mask_function,
    batched=False, # 必须逐条处理以正确计算 mask
    remove_columns=["text"]
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

Map:   0%|          | 0/5998 [00:00<?, ? examples/s]

# 4. LoRA Setup and Training

In [ ]:
import torch
import bitsandbytes as bnb

def suggest_target_modules(model):
    names = set()
    for full_name, module in model.named_modules():
        if isinstance(module, (torch.nn.Linear, bnb.nn.Linear4bit, bnb.nn.Linear8bitLt)):
            leaf = full_name.split(".")[-1]
            names.add(leaf)
    return sorted(names)

candidates = suggest_target_modules(model)
print("Linear-like leaf names in this model:", candidates)

Linear-like leaf names in this model: ['down_proj', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'q_proj', 'up_proj', 'v_proj']


In [ ]:
from peft import LoraConfig, get_peft_model

# Remove prior default adapter if any
try:
    model.delete_adapter("default")
except Exception:
    pass  # ignore if missing

# LoRA config in training mode; tune target_modules
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['base_layer', 'default', 'down_proj', 'gate_proj', 'lm_head', 'up_proj'],
    inference_mode=False
)

# Attach LoRA
model = get_peft_model(model, lora_config)

# With checkpointing, inputs must require grad
model.enable_input_require_grads()
model.train()

# Sanity: trainable count > 0
model.print_trainable_parameters()

trainable params: 28,889,088 || all params: 7,270,621,184 || trainable%: 0.3973


In [ ]:
# @title Pre-train smoke test
# 1) LoRA params trainable
model.train()
trainable = sum(p.requires_grad for p in model.parameters())
print("Trainable tensors count:", trainable)
model.print_trainable_parameters()

# 2) One batch: loss.requires_grad
from torch.utils.data import DataLoader
dl = DataLoader(tokenized, batch_size=1, collate_fn=data_collator)
batch = next(iter(dl))
batch = {k: v.to(model.device) for k, v in batch.items()}
out = model(**batch)
print("loss.requires_grad:", out.loss.requires_grad)

Trainable tensors count: 194
trainable params: 28,889,088 || all params: 7,270,621,184 || trainable%: 0.3973


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


loss.requires_grad: True


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=HF_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    torch_compile=False,
    report_to=["none"],
    gradient_checkpointing_kwargs={"use_reentrant": False},

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,1.211909
100,0.918394
150,0.902214
200,0.864418
250,0.849753
300,0.843101
350,0.830170


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


TrainOutput(global_step=375, training_loss=0.9118023885091145, metrics={'train_runtime': 3349.3337, 'train_samples_per_second': 1.791, 'train_steps_per_second': 0.112, 'total_flos': 2.2146950108440166e+17, 'train_loss': 0.9118023885091145, 'epoch': 1.0})

In [ ]:
# Save / push LoRA adapter
trainer.push_to_hub()

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  10%|9         | 64.0MB /  640MB            

  ...ll-v2.1/training_args.bin:  15%|#4        |   772B / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/imnotDawn/mistral7b-qlora-sft-small-v2.1/commit/cd07c3fae9752c1544447870a6bcb6f1c4564874', commit_message='End of training', commit_description='', oid='cd07c3fae9752c1544447870a6bcb6f1c4564874', pr_url=None, repo_url=RepoUrl('https://huggingface.co/imnotDawn/mistral7b-qlora-sft-small-v2.1', endpoint='https://huggingface.co', repo_type='model', repo_id='imnotDawn/mistral7b-qlora-sft-small-v2.1'), pr_revision=None, pr_num=None)

# 5. Inference and Evaluation

In [ ]:
from peft import PeftModel

# Reload base (4-bit) + adapter for inference
base_for_infer = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)
tok_for_infer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
peft_model = PeftModel.from_pretrained(base_for_infer, HF_OUTPUT_DIR)
peft_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
              (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
              (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
              (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
            )
            (mlp): MistralMLP(
              (gate_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=14336, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4

In [ ]:
def chat(messages, max_new_tokens=256, temperature=0.7):
    prompt = tok_for_infer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(peft_model.device)

    with torch.no_grad():
        output = peft_model.generate(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tok_for_infer.eos_token_id
        )
    return tok_for_infer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Empathy inference example
dialog = [
    {"role":"system","content":"You are an empathetic, practical mental-health assistant. Use your knowledge of cognitive behavioral therapy, meditation techniques, mindfulness practices, and other therapeutic methods in order to create strategies that the individual can implement in order to improve their overall wellbeing. Offer supportive reflection plus actionable next steps."},
    {"role":"user","content":"I've been feeling so sad and overwhelmed lately. Work has become such a massive source of stress for me."}
]

# Fix: apply_chat_template returns a BatchEncoding (dict), so we need to unpack it with **
inputs = tok_for_infer.apply_chat_template(
    dialog,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(peft_model.device)

with torch.no_grad():
    output = peft_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tok_for_infer.eos_token_id
    )

# Decode only the newly generated tokens (excluding the prompt)
input_length = inputs["input_ids"].shape[1]
response = tok_for_infer.decode(output[0][input_length:], skip_special_tokens=True)
print(response)

Friend, I can sense how deeply this is affecting you. It sounds like work-related stress has become overwhelming. Can you tell me more about what specifically is causing you to feel this way?
